# Serve Qianfan-OCR from Colab (vLLM + cloudflared tunnel)
Runtime: **GPU (T4 is enough)**. Run the cells top to bottom. The last cell prints the values for the worker:
`QIANFAN_URL` (base URL, no path) and `QIANFAN_REVISION`. The tunnel URL changes each session; never commit it.
With an API key set, also export `QIANFAN_API_KEY`. Anyone with the tunnel URL can use your GPU unless you set a key.

In [ ]:
# vLLM goes into its own virtual environment so it brings a consistent torch + cuDNN and does not mix with Colab's.
!pip install -q uv
!uv venv /content/vllm-env --python 3.12 -q
!uv pip install -q --python /content/vllm-env/bin/python vllm
!pip install -q huggingface_hub

In [ ]:
#@title Settings
REVISION = ""  #@param {type:"string"}
USE_API_KEY = True  #@param {type:"boolean"}
PORT = 8000

In [ ]:
# Resolve the revision to pin: the given one, else the current commit of the model repo (so you can record it).
from huggingface_hub import HfApi
MODEL = "baidu/Qianfan-OCR"
revision = REVISION.strip() or HfApi().model_info(MODEL).sha
print("revision:", revision)
api_key = None
if USE_API_KEY:
    from getpass import getpass
    api_key = getpass("API key to require (leave empty for none): ") or None

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
import subprocess, time, urllib.request, urllib.error, json, re, os, glob
VENV = "/content/vllm-env"
# Use the cuDNN that belongs to the venv's torch, ahead of anything Colab put on the library path.
cudnn_dirs = glob.glob(f"{VENV}/lib/python3.*/site-packages/nvidia/cudnn/lib")
env = dict(os.environ)
if cudnn_dirs:
    env["LD_LIBRARY_PATH"] = cudnn_dirs[0] + ":" + env.get("LD_LIBRARY_PATH", "")

# Fail fast (seconds, not after a 5-minute model download): can torch run a convolution on this GPU?
check = subprocess.run([f"{VENV}/bin/python", "-c",
    "import torch,torch.nn.functional as F;"
    "x=torch.randn(1,3,32,32,device='cuda',dtype=torch.float16);"
    "F.conv2d(x,torch.randn(8,3,3,3,device='cuda',dtype=torch.float16));"
    "print('conv2d ok, cudnn', torch.backends.cudnn.version(), 'torch', torch.__version__)"],
    env=env, capture_output=True, text=True)
print(check.stdout.strip() or check.stderr.strip()[-1500:])
if check.returncode != 0:
    raise RuntimeError("GPU convolution check failed (see above); do not start vLLM")

# T4 (16 GB, compute capability 7.5): a short context and no CUDA-graph capture keep memory use low.
cmd = [f"{VENV}/bin/vllm", "serve", MODEL, "--revision", revision, "--trust-remote-code",
       "--hf-overrides", json.dumps({"architectures": ["InternVLChatModel"]}),
       "--dtype", "float16", "--port", str(PORT),
       "--max-model-len", "8192", "--enforce-eager", "--gpu-memory-utilization", "0.90"]
if api_key:
    cmd += ["--api-key", api_key]
vllm = subprocess.Popen(cmd, env=env, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}

def log_tail(n=40):
    # the log is full of progress bars; show only the lines that matter
    lines = [l for l in open("vllm.log").read().splitlines() if l.strip() and "it/s]" not in l and "s/it]" not in l]
    return "\n".join(lines[-n:])

for _ in range(240):
    if vllm.poll() is not None:
        print(log_tail())
        raise RuntimeError("vLLM exited; the last log lines are above")
    try:
        urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/models", headers=headers), timeout=2)
        break
    except urllib.error.HTTPError:
        break
    except Exception:
        time.sleep(5)
else:
    print(log_tail())
    raise RuntimeError("vLLM did not come up in 20 minutes; the last log lines are above")
print("vLLM is up")

In [ ]:
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stderr=subprocess.PIPE, text=True)
url = None
for line in tunnel.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print("Paste into the worker environment:")
print(f"  QIANFAN_URL={url}")
print(f"  QIANFAN_REVISION={revision}")
if api_key:
    print("  QIANFAN_API_KEY=<the key you entered>")
print("  EXTRACTOR=http")